# Corpus audit — semi-automatic noise detection (human verifies, never searches)

The reusability matrix prices per-topic corpus curation at "half a day of manual
inspection". This notebook replaces that with a **semi-automatic auditor**: two generic
detection channels produce a **ranked shortlist**, and the human's only job is a YES/NO
pass over ~40 rows — flagging is automated, verification is not (and should not be).

- **Channel 1 — metadata structure scan** (zero topic knowledge): distribution tables over
  the structural fields every source carries (RIS senate codes & Rechtsgebiete, Swiss court
  types & title patterns, ECHR doctypes). Noise families are usually *structurally* visible
  before they are semantically visible.
- **Channel 2 — embedding outlier ranking**: every document scored by cosine of its mean
  chunk vector against the corpus centroid; the bottom tail is the candidate-noise
  shortlist. Uses the cached vectors; excluded documents are embedded on the fly (one
  window each) so the audit sees the *unfiltered* corpus.
- **Validation on known ground truth**: this topic's three noise families were found
  manually (criminal-senate *Entfremdung*, AUSL/Bsw summaries, Swiss Rechenschaftsberichte).
  The auditor runs blind — if the known noise concentrates in its bottom tail, the
  "verify-only" workflow is justified for the next topic.

**Topic-agnostic**: no lexicon, no rules — structure and geometry only.
Output: `data/corpus_audit_shortlist.csv` (blank `exclude` column = the human's only task).

## Channel 1 — metadata structure scan

In [ ]:
import json
import re
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd

DATA_DIR, REPORT_DIR = Path("../data"), Path("../reports")
ris   = json.loads((DATA_DIR / "ris_parental_alienation.json").read_text())
swiss = json.loads((DATA_DIR / "swiss_parental_alienation.json").read_text())
echr  = json.loads((DATA_DIR / "echr_parental_alienation.json").read_text())

_SEN = re.compile(r"\d{1,3}\s*([A-Z][a-z]{1,2})")
def senate(gz):
    m = _SEN.match((gz or "").split(";")[0].strip())
    return m.group(1) if m else "?"

print("RIS by senate code (structure a human scans in seconds):")
print(pd.Series(Counter(senate(r.get("geschaeftszahl")) for r in ris)).sort_values(ascending=False).to_string())
print("\nRIS by Rechtsgebiet:")
print(pd.Series(Counter(r.get("rechtsgebiete") or "(none)" for r in ris)).sort_values(ascending=False).head(6).to_string())
print("\nSwiss title tokens that are NOT case references (top non-standard patterns):")
title_flags = Counter()
for r in swiss:
    t = str(r.get("title") or "")
    for tok in ("Rechenschaftsbericht", "Kreisschreiben", "Reglement", "Verordnung", "Bericht"):
        if tok.lower() in t.lower():
            title_flags[tok] += 1
print(pd.Series(title_flags).to_string() if title_flags else "  (none)")
print("\nECHR by doctype:", dict(Counter(r.get("doctype") for r in echr)))

RIS by senate code (structure a human scans in seconds):
Ob    508
Os     33
?       7

RIS by Rechtsgebiet:
(none)        510
Zivilrecht     29
Strafrecht      8
Undefined       1

Swiss title tokens that are NOT case references (top non-standard patterns):
Rechenschaftsbericht    24
Bericht                 27

ECHR by doctype: {'HEDEC': 311, 'HEJUD': 567, 'HECOM': 238}


## Channel 2 — embedding outlier ranking over the *unfiltered* corpus

In [ ]:
cache = np.load(DATA_DIR / "rag_echr_ris_emb_cache.npz", allow_pickle=True)
ids = [str(x) for x in cache["ids"]]
emb = cache["emb"].astype("float32")

# doc-level vectors for everything already indexed (mean of its chunks)
doc_rows = {}
for i, cid in enumerate(ids):
    src, doc_id = cid.split(":")[0], cid.split(":")[1]
    doc_rows.setdefault((src, doc_id), []).append(i)

doc_vec, doc_meta = {}, {}
def norm(v):
    return v / (np.linalg.norm(v) + 1e-9)
for key, rows in doc_rows.items():
    doc_vec[key] = norm(emb[rows].mean(axis=0))

# embed the EXCLUDED documents on the fly (one 600-word window each) so the audit is blind
_WS = re.compile(r"\S+")
def first_window(text, size=600):
    w = _WS.findall(text or "")
    return " ".join(w[:size])

excluded = []
for r in ris:
    if r.get("dokumenttyp") == "Text":
        gz = (r.get("geschaeftszahl") or "").split(";")[0]
        if "OGH0002" not in (r.get("id") or "") or re.search(r"\d{1,3}\s*Os\s*\d", gz):
            excluded.append(("ris", r.get("id"), r.get("full_text"), "known:criminal/AUSL"))
for r in swiss:
    if "rechenschaftsbericht" in str(r.get("title") or "").lower():
        excluded.append(("swiss", r.get("stable_id"), r.get("content"), "known:annual-report"))
print(f"excluded docs to embed on the fly: {len(excluded)}")

from sentence_transformers import SentenceTransformer
model = SentenceTransformer("intfloat/multilingual-e5-base", device="cpu")
texts = [f"passage: {first_window(t)}" for _, _, t, _ in excluded]
if texts:
    ex_emb = model.encode(texts, batch_size=8, convert_to_numpy=True,
                          normalize_embeddings=True, show_progress_bar=False).astype("float32")
    for (src, did, _, flag), v in zip(excluded, ex_emb):
        doc_vec[(src, did)] = v

known_noise = {(src, did) for src, did, _, flag in excluded}
print(f"total documents scored: {len(doc_vec)}")

excluded docs to embed on the fly: 55
/Users/maksimsmirnov/Desktop/thesis/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
total documents scored: 3697


In [ ]:
# corpus centroid + outlier ranking (ascending similarity = most anomalous first)
mat = np.stack(list(doc_vec.values()))
keys = list(doc_vec.keys())
centroid = norm(mat.mean(axis=0))
sims = mat @ centroid
order = np.argsort(sims)

# titles/meta for display
title_by = {}
for r in ris:
    title_by[("ris", r.get("id"))] = f"{r.get('gericht','OGH')} {(r.get('geschaeftszahl') or '')[:24]} [{r.get('rechtsgebiete') or r.get('dokumenttyp')}]"
for r in swiss:
    title_by[("swiss", r.get("stable_id"))] = str(r.get("title"))[:60]
for r in echr:
    title_by[("echr", r.get("itemid"))] = str(r.get("docname"))[:60]

# VALIDATION: where does the KNOWN noise rank, blind?
pct = {k: 100 * np.searchsorted(sims[order], sims[i]) / len(keys)
       for i, k in enumerate(keys)}
noise_pcts = [pct[k] for k in known_noise if k in pct]
print(f"known-noise docs: {len(noise_pcts)} | median outlier percentile "
      f"{np.median(noise_pcts):.0f} (0 = most anomalous)")
print(f"share of known noise in the bottom 10% of the ranking: "
      f"{np.mean([p <= 10 for p in noise_pcts])*100:.0f}%")
print(f"(random expectation would be 10%)\n")

print("BOTTOM 20 of the blind outlier ranking (candidate noise; * = known noise family):")
for i in order[:20]:
    k = keys[i]
    star = "*" if k in known_noise else " "
    print(f" {star} cos={sims[i]:.3f} | {k[0]:5s} | {title_by.get(k, k[1])[:70]}")

known-noise docs: 55 | median outlier percentile 3 (0 = most anomalous)
share of known noise in the bottom 10% of the ranking: 100%
(random expectation would be 10%)

BOTTOM 20 of the blind outlier ranking (candidate noise; * = known noise family):
   cos=0.835 | ris   | OGH 14Os46/90; 15Os56/92; 13 [Strafrecht]
   cos=0.838 | ris   | OGH 11Os59/75; 10Os18/76; 11 [Strafrecht]
   cos=0.838 | ris   | OGH 6Ob310/66; 8Ob47/67; 6Ob [Zivilrecht]
   cos=0.839 | ris   | OGH 9Os153/81; 10Os101/82; 1 [Strafrecht]
   cos=0.843 | ris   | OGH 1Ob60/71; 1Ob201/72 (1Ob [Zivilrecht]
   cos=0.845 | ris   | OGH 7Ob259/75; 7Ob514/76; 3O [Zivilrecht]
   cos=0.851 | ris   | OGH 4Ob538/92; 1Ob7/00m; 7Ob [Zivilrecht]
   cos=0.851 | ris   | OGH 13Os176/80; 13Os148/82;  [Strafrecht]
   cos=0.851 | ris   | OGH 12Os43/76; 13Os29/80; 13 [Strafrecht]
   cos=0.855 | ris   | OGH 6Ob193/73; 5Ob670/79; 8O [Zivilrecht]
   cos=0.858 | ris   | OGH 6Ob63/18k; 6Ob13/23i [Zivilrecht]
   cos=0.859 | ris   | OGH 5Ob15/60; 1Ob

## Shortlist for human verification (the only manual step)

In [ ]:
rows = []
for i in order[:40]:
    k = keys[i]
    rows.append({"source": k[0], "doc_id": k[1], "similarity_to_corpus": round(float(sims[i]), 4),
                 "title": title_by.get(k, ""), "known_noise": k in known_noise,
                 "exclude": ""})
shortlist = pd.DataFrame(rows)
out = DATA_DIR / "corpus_audit_shortlist.csv"
shortlist.to_csv(out, index=False)
print(f"wrote {out.name}: 40 rows, human fills `exclude` (YES/NO) — nothing else")

lines = ["# Corpus audit report (semi-automatic curation)\n\n"]
lines.append(f"- documents scored (unfiltered corpus): **{len(doc_vec)}**; known noise from "
             f"the manual round: {len(noise_pcts)} docs.\n")
lines.append(f"- **blind recovery**: median outlier percentile of known noise "
             f"**{np.median(noise_pcts):.0f}** (0 = most anomalous); "
             f"**{np.mean([p <= 10 for p in noise_pcts])*100:.0f}%** of known noise sits in "
             f"the bottom 10% of the ranking (random baseline: 10%).\n")
lines.append("- workflow for a NEW topic: run this notebook -> verify the 40-row shortlist "
             "(YES/NO) -> add confirmed exclusions to the corpus-build filters. Flagging is "
             "automated; only verification is human.\n")
lines.append("- channels are topic-agnostic: metadata structure scan + embedding outlier "
             "geometry; no lexicons, no rules.\n")
(REPORT_DIR / "corpus_audit_report.md").write_text("".join(lines), encoding="utf-8")
print("wrote", REPORT_DIR / "corpus_audit_report.md")

wrote corpus_audit_shortlist.csv: 40 rows, human fills `exclude` (YES/NO) — nothing else
wrote ../reports/corpus_audit_report.md
